# Machine Learning models

In this notebook we focus on training machine learning models to separate groups with CD vs nonIBD and UC vs nonIBD based only on the microbiome data.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             precision_score, recall_score,
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

# Load data
meta = pd.read_csv("../data/processed/metadata_final.tsv", sep="\t", index_col="Sample")
genera_clr = pd.read_csv("../data/processed/genera_clr.tsv", sep="\t", index_col=0)
meta = meta.loc[genera_clr.index]

print("Metadata shape:", meta.shape)
print("Genera CLR shape:", genera_clr.shape)
print("\nDiagnosis counts:")
print(meta["Study.Group"].value_counts())

Metadata shape: (105, 23)
Genera CLR shape: (105, 2761)

Diagnosis counts:
Study.Group
CD        49
UC        30
nonIBD    26
Name: count, dtype: int64


## Prepare two binary datasets

For the following of the analysis we create two completely separate datasets. Each has the same genera columns but different rows (subjects). The labels are binary: 1 for the IBD group and 0 for the nonIBD. We note again that the two classifiaction problems are independent.

In [3]:
def make_binary_dataset(meta, genera, group_a, group_b):
    """
    Filters metadata and genera to only include two groups.
    Returns X (features) and y (binary labels).
    """
    mask = meta["Study.Group"].isin([group_a, group_b])
    meta_sub = meta[mask]
    X = genera.loc[meta_sub.index]
    y = (meta_sub["Study.Group"] == group_a).astype(int)
    # 1 = group_a, 0 = group_b
    print(f"\n{group_a} vs {group_b}")
    print(f"  {group_a}: {(y==1).sum()} samples")
    print(f"  {group_b}: {(y==0).sum()} samples")
    print(f"  Total: {len(y)} samples")
    return X, y

X_uc, y_uc = make_binary_dataset(meta, genera_clr, "UC", "nonIBD")
X_cd, y_cd = make_binary_dataset(meta, genera_clr, "CD", "nonIBD")


UC vs nonIBD
  UC: 30 samples
  nonIBD: 26 samples
  Total: 56 samples

CD vs nonIBD
  CD: 49 samples
  nonIBD: 26 samples
  Total: 75 samples


## Feature Selection with Kruskal-Wallis filter

We keep the top 100 genera.

In [4]:
def kruskal_feature_selection(X, y, top_k):
    """
    Runs Kruskal-Wallis test for each genus between the two groups.
    Returns the names of the top_k most significant genera.
    """
    group0 = X[y == 0]
    group1 = X[y == 1]
    
    p_values = []
    for col in X.columns:
        stat, p = kruskal(group0[col].values, group1[col].values)
        p_values.append(p)
    
    p_series = pd.Series(p_values, index=X.columns)
    top_features = p_series.nsmallest(top_k).index.tolist()
    return top_features, p_series

# Run for both tasks, keep top 100 genera
print("=== UC vs nonIBD ===")
uc_features, uc_pvals = kruskal_feature_selection(X_uc, y_uc, top_k=100)
print(f"Top 100 genera selected by Kruskal-Wallis")
print(f"Best p-value: {uc_pvals.min():.4f}")

print("\n=== CD vs nonIBD ===")
cd_features, cd_pvals = kruskal_feature_selection(X_cd, y_cd, top_k=100)
print(f"Top 100 genera selected by Kruskal-Wallis")
print(f"Best p-value: {cd_pvals.min():.4f}")

=== UC vs nonIBD ===
Top 100 genera selected by Kruskal-Wallis
Best p-value: 0.0013

=== CD vs nonIBD ===
Top 100 genera selected by Kruskal-Wallis
Best p-value: 0.0007


## Feature Selection with PCA

We retain the 80% of the variance.

In [5]:
def pca_feature_selection(X, variance_threshold=0.80):
    """
    Applies PCA and keeps enough components to explain
    the target variance threshold.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    pca = PCA(n_components=min(X.shape[0]-1, X.shape[1]))
    pca.fit(X_scaled)
    
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n_components = np.argmax(cumvar >= variance_threshold) + 1
    
    print(f"Components to explain {variance_threshold*100:.0f}% variance: {n_components}")
    print(f"Variance explained by first 10 PCs: {cumvar[9]*100:.1f}%")
    
    # Refit with correct number of components
    pca_final = PCA(n_components=n_components)
    X_pca = pca_final.fit_transform(X_scaled)
    
    return X_pca, pca_final, scaler, n_components

print("=== UC vs nonIBD ===")
X_uc_pca, pca_uc, scaler_uc, n_uc = pca_feature_selection(X_uc)

print("\n=== CD vs nonIBD ===")
X_cd_pca, pca_cd, scaler_cd, n_cd = pca_feature_selection(X_cd)

=== UC vs nonIBD ===
Components to explain 80% variance: 32
Variance explained by first 10 PCs: 47.6%

=== CD vs nonIBD ===
Components to explain 80% variance: 41
Variance explained by first 10 PCs: 44.4%


# Comments on the features

Both binary classification tasks seem to have modest biological signal. We do not expect a perfect classification, but a solid one that could improve adding more parameters to the process (like age, gender or BMI).